In [ ]:
!pip install -q opencv-python-headless pandas numpy matplotlib tqdm pillow

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

In [ ]:
# Local extraction path from the uploaded zip
ROOT = Path("/content/extracted_dataset")

# Paths based on the extracted zip structure
IMAGE_DIR = ROOT / "images"
MASK_DIR = ROOT / "masks"

# Output directories for chlorosis analysis
OUTPUT_DIR = Path("/content/chlorosis_outputs")
CHLOROSIS_MASK_DIR = OUTPUT_DIR / "chlorosis_masks"
OVERLAY_DIR = OUTPUT_DIR / "chlorosis_overlays"
SCORE_DIR = OUTPUT_DIR / "chlorosis_score_maps"
META_DIR = OUTPUT_DIR / "metadata"

# Create output folders
for d in [OUTPUT_DIR, CHLOROSIS_MASK_DIR, OVERLAY_DIR, SCORE_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Source Images:", IMAGE_DIR)
print("Source Masks :", MASK_DIR)
print("Output Folder:", OUTPUT_DIR)

In [ ]:
def get_chlorosis_score(image_rgb, mask):
    # Convert to Lab to look for yellowing/lightness shifts
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    # Chlorosis often shows as higher b* (yellow) and higher L* (brightness)
    # Normalize b_channel to 0-1
    score_map = b_channel.astype(float) / 255.0
    # Only keep scores where the lichen mask is active
    score_map[mask == 0] = 0
    return score_map

results = []
image_files = list(IMAGE_DIR.glob('*.jpg'))

print(f'Processing {len(image_files)} images...')
for img_path in tqdm(image_files):
    # Find corresponding mask
    mask_name = f'{img_path.stem}_lichen_mask.png'
    mask_path = MASK_DIR / mask_name

    if not mask_path.exists():
        continue

    # Load image and mask
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None: continue

    # Calculate score
    score_map = get_chlorosis_score(img, mask)

    # Simple threshold for mask visualization
    chlorosis_mask = (score_map > 0.6).astype(np.uint8) * 255

    # Save outputs
    cv2.imwrite(str(CHLOROSIS_MASK_DIR / f'{img_path.stem}_chlorosis.png'), chlorosis_mask)

    results.append({
        'filename': img_path.name,
        'avg_chlorosis_score': np.mean(score_map[mask > 0]) if np.any(mask > 0) else 0
    })

df_results = pd.DataFrame(results)
df_results.to_csv(META_DIR / 'chlorosis_results.csv', index=False)
print('\nAnalysis complete. Results saved to metadata.')
df_results.head()

In [ ]:
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"]

def clean_stem(path):
    stem = Path(path).stem.lower()

    remove_terms = [
        "_lichen_mask",
        "-lichen-mask",
        "_coverage_mask",
        "-coverage-mask",
        "_auto_mask",
        "-auto-mask",
        "_mask",
        "-mask",
        "_binary",
        "-binary",
        "_overlay",
        "-overlay"
    ]

    for term in remove_terms:
        stem = stem.replace(term, "")

    return stem


def collect_images(folder):
    files = []
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            files.append(p)
    return sorted(files)


image_files = collect_images(IMAGE_DIR)
mask_files = collect_images(MASK_DIR)

print("Images found:", len(image_files))
print("Masks found :", len(mask_files))

image_dict = {clean_stem(p): p for p in image_files}

pairs = []

for mask_path in mask_files:
    key = clean_stem(mask_path)
    if key in image_dict:
        pairs.append((image_dict[key], mask_path))

print("Matched image-mask pairs:", len(pairs))

for img, mask in pairs[:10]:
    print("IMAGE:", img.name)
    print("MASK :", mask.name)
    print()

In [ ]:
def load_rgb(image_path):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def load_binary_mask(mask_path):
    raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if raw is None:
        raise ValueError(f"Could not read mask: {mask_path}")

    if len(raw.shape) == 2:
        gray = raw
    elif raw.shape[2] == 4:
        # If alpha channel is meaningful, use it.
        b, g, r, a = cv2.split(raw)

        if np.max(a) > 0 and np.min(a) == 0:
            gray = a
        else:
            gray = cv2.cvtColor(raw[:, :, :3], cv2.COLOR_BGR2GRAY)
    else:
        gray = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY)

    mask = (gray > 127).astype(np.uint8)
    return mask


def resize_mask_to_image(mask, rgb):
    if mask.shape[:2] != rgb.shape[:2]:
        mask = cv2.resize(
            mask,
            (rgb.shape[1], rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )
    return mask.astype(np.uint8)

In [ ]:
class LichenChlorosisModel:
    def __init__(
        self,
        chlorosis_threshold=0.62,
        bleach_threshold=0.68,
        min_region_area=40
    ):
        self.chlorosis_threshold = chlorosis_threshold
        self.bleach_threshold = bleach_threshold
        self.min_region_area = min_region_area

    def normalize_inside_mask(self, channel, mask, low_pct=5, high_pct=95):
        values = channel[mask == 1]

        if len(values) == 0:
            return np.zeros_like(channel, dtype=np.float32)

        low = np.percentile(values, low_pct)
        high = np.percentile(values, high_pct)

        norm = (channel - low) / (high - low + 1e-6)
        norm = np.clip(norm, 0, 1)

        return norm.astype(np.float32)

    def remove_small_regions(self, mask, min_area=None):
        if min_area is None:
            min_area = self.min_region_area

        mask = mask.astype(np.uint8)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )

        cleaned = np.zeros_like(mask, dtype=np.uint8)

        for label in range(1, num_labels):
            area = stats[label, cv2.CC_STAT_AREA]

            if area >= min_area:
                cleaned[labels == label] = 1

        return cleaned

    def smooth_score(self, score):
        score_uint8 = np.clip(score * 255, 0, 255).astype(np.uint8)
        score_blur = cv2.GaussianBlur(score_uint8, (5, 5), 0)
        return score_blur.astype(np.float32) / 255.0

    def compute_features(self, rgb, lichen_mask):
        rgb_float = rgb.astype(np.float32)

        R = rgb_float[:, :, 0]
        G = rgb_float[:, :, 1]
        B_rgb = rgb_float[:, :, 2]

        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

        L = lab[:, :, 0].astype(np.float32)
        A = lab[:, :, 1].astype(np.float32)
        B_lab = lab[:, :, 2].astype(np.float32)

        H = hsv[:, :, 0].astype(np.float32)
        S = hsv[:, :, 1].astype(np.float32)
        V = hsv[:, :, 2].astype(np.float32)

        # LAB yellow axis: OpenCV LAB B > 128 means more yellow.
        yellow_lab = np.maximum(B_lab - 128, 0)

        # RGB yellowing index.
        yellow_rgb = (R + G - 2 * B_rgb) / (R + G + B_rgb + 1e-6)

        # Lack of greenness.
        excess_green = 2 * G - R - B_rgb
        lack_green = -excess_green

        # Bleaching / paleness: high brightness, low saturation.
        paleness = L
        low_saturation = 255 - S

        # Yellow hue confidence.
        # OpenCV hue range is 0–179. Yellow/orange roughly 15–45.
        yellow_hue_conf = np.zeros_like(H, dtype=np.float32)
        yellow_hue_conf[(H >= 14) & (H <= 48)] = 1.0

        # Brown / dead-looking tissue.
        brown_hue_conf = np.zeros_like(H, dtype=np.float32)
        brown_hue_conf[(H >= 5) & (H <= 25) & (S > 40) & (V < 190)] = 1.0

        # Local color abnormality inside the lichen region.
        lichen_lab = lab[lichen_mask == 1].astype(np.float32)

        if len(lichen_lab) > 0:
            median_lab = np.median(lichen_lab, axis=0)
            delta_lab = np.sqrt(
                (L - median_lab[0]) ** 2 +
                (A - median_lab[1]) ** 2 +
                (B_lab - median_lab[2]) ** 2
            )
        else:
            delta_lab = np.zeros_like(L)

        features = {
            "yellow_lab": self.normalize_inside_mask(yellow_lab, lichen_mask),
            "yellow_rgb": self.normalize_inside_mask(yellow_rgb, lichen_mask),
            "lack_green": self.normalize_inside_mask(lack_green, lichen_mask),
            "paleness": self.normalize_inside_mask(paleness, lichen_mask),
            "low_saturation": self.normalize_inside_mask(low_saturation, lichen_mask),
            "local_abnormality": self.normalize_inside_mask(delta_lab, lichen_mask),
            "yellow_hue_conf": yellow_hue_conf,
            "brown_hue_conf": brown_hue_conf
        }

        return features

    def predict(self, rgb, lichen_mask):
        lichen_mask = lichen_mask.astype(np.uint8)

        if lichen_mask.sum() == 0:
            empty = np.zeros(lichen_mask.shape, dtype=np.uint8)
            score = np.zeros(lichen_mask.shape, dtype=np.float32)

            return {
                "chlorosis_mask": empty,
                "chlorosis_score": score,
                "chlorosis_percent": 0.0,
                "severity": "invalid_no_lichen_mask",
                "metrics": {}
            }

        f = self.compute_features(rgb, lichen_mask)

        # Main yellow/chlorosis score.
        yellowing_score = (
            0.40 * f["yellow_lab"] +
            0.25 * f["yellow_rgb"] +
            0.20 * f["lack_green"] +
            0.15 * f["local_abnormality"]
        )

        # Make yellowing stronger when hue is actually yellow.
        yellowing_score = yellowing_score * (0.65 + 0.35 * f["yellow_hue_conf"])

        # Bleaching score catches white/pale damaged tissue.
        bleaching_score = (
            0.45 * f["paleness"] +
            0.35 * f["low_saturation"] +
            0.20 * f["local_abnormality"]
        )

        # Brown/damaged score.
        brown_damage_score = (
            0.55 * f["brown_hue_conf"] +
            0.25 * f["local_abnormality"] +
            0.20 * f["lack_green"]
        )

        # Combined chlorosis/stress discoloration score.
        chlorosis_score = (
            0.50 * yellowing_score +
            0.35 * bleaching_score +
            0.15 * brown_damage_score
        )

        chlorosis_score = chlorosis_score * lichen_mask
        chlorosis_score = self.smooth_score(chlorosis_score)
        chlorosis_score = chlorosis_score * lichen_mask

        # Pixel-level decision.
        chlorosis_mask = (
            ((chlorosis_score > self.chlorosis_threshold) |
             (bleaching_score > self.bleach_threshold)) &
            (lichen_mask == 1)
        ).astype(np.uint8)

        chlorosis_mask = self.remove_small_regions(chlorosis_mask)

        lichen_pixels = int(lichen_mask.sum())
        chlorosis_pixels = int(chlorosis_mask.sum())

        chlorosis_percent = 100.0 * chlorosis_pixels / max(lichen_pixels, 1)

        severity = self.classify_severity(chlorosis_percent)

        metrics = {
            "lichen_pixels": lichen_pixels,
            "chlorosis_pixels": chlorosis_pixels,
            "chlorosis_percent": float(chlorosis_percent),

            "mean_chlorosis_score_inside_lichen": float(chlorosis_score[lichen_mask == 1].mean()),
            "p90_chlorosis_score_inside_lichen": float(np.percentile(chlorosis_score[lichen_mask == 1], 90)),
            "mean_yellowing_score_inside_lichen": float(yellowing_score[lichen_mask == 1].mean()),
            "mean_bleaching_score_inside_lichen": float(bleaching_score[lichen_mask == 1].mean()),
            "mean_brown_damage_score_inside_lichen": float(brown_damage_score[lichen_mask == 1].mean()),
        }

        return {
            "chlorosis_mask": chlorosis_mask,
            "chlorosis_score": chlorosis_score,
            "yellowing_score": yellowing_score * lichen_mask,
            "bleaching_score": bleaching_score * lichen_mask,
            "brown_damage_score": brown_damage_score * lichen_mask,
            "chlorosis_percent": float(chlorosis_percent),
            "severity": severity,
            "metrics": metrics
        }

    def classify_severity(self, chlorosis_percent):
        if chlorosis_percent < 5:
            return "minimal"
        elif chlorosis_percent < 15:
            return "low"
        elif chlorosis_percent < 35:
            return "moderate"
        else:
            return "high"

In [ ]:
def make_chlorosis_overlay(rgb, lichen_mask, chlorosis_mask):
    overlay = rgb.copy()
    color_layer = rgb.copy()

    # Dim background outside lichen slightly.
    dimmed = (overlay * 0.55).astype(np.uint8)
    overlay[lichen_mask == 0] = dimmed[lichen_mask == 0]

    # Yellow overlay for chlorosis.
    color_layer[chlorosis_mask == 1] = [255, 230, 0]

    blended = cv2.addWeighted(overlay, 0.68, color_layer, 0.32, 0)
    return blended


def save_score_map(score, out_path):
    score_uint8 = np.clip(score * 255, 0, 255).astype(np.uint8)
    heat = cv2.applyColorMap(score_uint8, cv2.COLORMAP_JET)
    cv2.imwrite(str(out_path), heat)


def show_result(rgb, lichen_mask, result, title="Chlorosis result"):
    overlay = make_chlorosis_overlay(
        rgb,
        lichen_mask,
        result["chlorosis_mask"]
    )

    plt.figure(figsize=(16, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(rgb)
    plt.axis("off")
    plt.title("Image")

    plt.subplot(1, 4, 2)
    plt.imshow(lichen_mask, cmap="gray")
    plt.axis("off")
    plt.title("Lichen mask")

    plt.subplot(1, 4, 3)
    plt.imshow(result["chlorosis_score"], cmap="inferno")
    plt.axis("off")
    plt.title("Chlorosis score")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(
        f"{result['chlorosis_percent']:.2f}% | {result['severity']}"
    )

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
model = LichenChlorosisModel(
    chlorosis_threshold=0.7400000000000001,
    bleach_threshold=0.7800000000000001,
    min_region_area=80
)

TEST_INDEX = 0

image_path, mask_path = pairs[TEST_INDEX]

rgb = load_rgb(image_path)
lichen_mask = load_binary_mask(mask_path)
lichen_mask = resize_mask_to_image(lichen_mask, rgb)

result = model.predict(rgb, lichen_mask)

print("Image:", image_path.name)
print("Mask :", mask_path.name)
print("Chlorosis %:", result["chlorosis_percent"])
print("Severity:", result["severity"])
print(result["metrics"])

show_result(rgb, lichen_mask, result, title=image_path.name)

In [ ]:
model = LichenChlorosisModel(
    chlorosis_threshold=0.7400000000000001,
    bleach_threshold=0.7800000000000001,
    min_region_area=80
)

records = []

for image_path, mask_path in tqdm(pairs):
    try:
        rgb = load_rgb(image_path)
        lichen_mask = load_binary_mask(mask_path)
        lichen_mask = resize_mask_to_image(lichen_mask, rgb)

        result = model.predict(rgb, lichen_mask)

        image_id = image_path.stem

        chlorosis_mask_out = CHLOROSIS_MASK_DIR / f"{image_id}_chlorosis_mask.png"
        overlay_out = OVERLAY_DIR / f"{image_id}_chlorosis_overlay.png"
        score_out = SCORE_DIR / f"{image_id}_chlorosis_score.png"

        cv2.imwrite(str(chlorosis_mask_out), result["chlorosis_mask"] * 255)

        overlay = make_chlorosis_overlay(
            rgb,
            lichen_mask,
            result["chlorosis_mask"]
        )
        cv2.imwrite(str(overlay_out), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

        save_score_map(result["chlorosis_score"], score_out)

        m = result["metrics"]

        records.append({
            "image_id": image_id,
            "image_path": str(image_path),
            "lichen_mask_path": str(mask_path),
            "chlorosis_mask_path": str(chlorosis_mask_out),
            "chlorosis_overlay_path": str(overlay_out),
            "chlorosis_score_map_path": str(score_out),

            "chlorosis_percent": result["chlorosis_percent"],
            "chlorosis_severity": result["severity"],

            "lichen_pixels": m.get("lichen_pixels", 0),
            "chlorosis_pixels": m.get("chlorosis_pixels", 0),
            "mean_chlorosis_score_inside_lichen": m.get("mean_chlorosis_score_inside_lichen", np.nan),
            "p90_chlorosis_score_inside_lichen": m.get("p90_chlorosis_score_inside_lichen", np.nan),
            "mean_yellowing_score_inside_lichen": m.get("mean_yellowing_score_inside_lichen", np.nan),
            "mean_bleaching_score_inside_lichen": m.get("mean_bleaching_score_inside_lichen", np.nan),
            "mean_brown_damage_score_inside_lichen": m.get("mean_brown_damage_score_inside_lichen", np.nan),

            "model_name": "lichen_gated_lab_hsv_chlorosis_index_v1",
            "chlorosis_threshold": model.chlorosis_threshold,
            "bleach_threshold": model.bleach_threshold,
            "error": ""
        })

    except Exception as e:
        records.append({
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "lichen_mask_path": str(mask_path),
            "error": str(e)
        })

chlorosis_df = pd.DataFrame(records)

out_csv = META_DIR / "chlorosis_predictions.csv"
chlorosis_df.to_csv(out_csv, index=False)

print("Saved:", out_csv)
display(chlorosis_df.head())

print("\nSeverity counts:")
print(chlorosis_df["chlorosis_severity"].value_counts(dropna=False))

In [ ]:
def review_chlorosis_outputs(df, n=12, seed=None, severity=None):
    """Displays a grid of chlorosis overlays for visual review."""
    temp_df = df.copy()
    if severity:
        temp_df = temp_df[temp_df['chlorosis_severity'] == severity]

    n = min(n, len(temp_df))
    if n == 0:
        print(f"No samples found for severity: {severity}")
        return

    sample = temp_df.sample(n=n, random_state=seed)

    cols = 4
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(20, 5 * rows))

    for i, (_, row) in enumerate(sample.iterrows()):
        img_path = row['chlorosis_overlay_path']
        if Path(img_path).exists():
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.title(f"{row['image_id'][:20]}\n{row['chlorosis_percent']:.1f}% ({row['chlorosis_severity']})")
            plt.axis('off')

    plt.tight_layout()
    plt.show()

# Run the review
review_chlorosis_outputs(chlorosis_df, n=12, seed=2, severity="high")

Auto-Adjust Threshold

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def infer_label_group(image_path):
    name = image_path.stem.lower()

    if "natural_yellow" in name or "natural_y" in name or "orange" in name or "xanthoria" in name or "candelaria" in name:
        return "natural_yellow_control"

    if "healthy" in name:
        return "healthy_control"

    if "positive" in name or "bleach" in name or "chlorosis" in name or "discolor" in name:
        return "positive_candidate"

    return "unknown"


def run_threshold_sweep(pairs, thresholds=None, bleach_offset=0.04, max_images=None):
    if thresholds is None:
        thresholds = np.arange(0.62, 0.91, 0.02)

    if max_images is not None:
        pairs_to_use = pairs[:max_images]
    else:
        pairs_to_use = pairs

    rows = []

    for threshold in thresholds:
        bleach_threshold = min(threshold + bleach_offset, 0.95)

        model = LichenChlorosisModel(
            chlorosis_threshold=float(threshold),
            bleach_threshold=float(bleach_threshold),
            min_region_area=80
        )

        for image_path, mask_path in tqdm(pairs_to_use, desc=f"threshold={threshold:.2f}"):
            try:
                rgb = load_rgb(image_path)
                lichen_mask = load_binary_mask(mask_path)
                lichen_mask = resize_mask_to_image(lichen_mask, rgb)

                result = model.predict(rgb, lichen_mask)

                rows.append({
                    "threshold": float(threshold),
                    "bleach_threshold": float(bleach_threshold),
                    "image_id": image_path.stem,
                    "label_group": infer_label_group(image_path),
                    "chlorosis_percent": result["chlorosis_percent"],
                    "severity": result["severity"]
                })

            except Exception as e:
                rows.append({
                    "threshold": float(threshold),
                    "bleach_threshold": float(bleach_threshold),
                    "image_id": image_path.stem,
                    "label_group": infer_label_group(image_path),
                    "chlorosis_percent": np.nan,
                    "severity": "error",
                    "error": str(e)
                })

    return pd.DataFrame(rows)


sweep_df = run_threshold_sweep(
    pairs,
    thresholds=np.arange(0.62, 0.91, 0.02),
    bleach_offset=0.04,
    max_images=None
)

display(sweep_df.head())

In [ ]:
def pick_best_threshold(sweep_df):
    df = sweep_df.dropna(subset=["chlorosis_percent"]).copy()

    negative_groups = ["healthy_control", "natural_yellow_control"]

    neg = df[df["label_group"].isin(negative_groups)].copy()

    if len(neg) == 0:
        print("No healthy/natural-yellow controls found. Defaulting to 0.78.")
        return 0.78, 0.82

    summary = []

    for threshold, sub in neg.groupby("threshold"):
        median_neg = sub["chlorosis_percent"].median()
        p75_neg = sub["chlorosis_percent"].quantile(0.75)
        p90_neg = sub["chlorosis_percent"].quantile(0.90)
        false_high_rate = (sub["chlorosis_percent"] >= 20).mean()
        false_moderate_rate = (sub["chlorosis_percent"] >= 15).mean()

        summary.append({
            "threshold": threshold,
            "negative_median_percent": median_neg,
            "negative_p75_percent": p75_neg,
            "negative_p90_percent": p90_neg,
            "false_high_rate": false_high_rate,
            "false_moderate_rate": false_moderate_rate
        })

    summary_df = pd.DataFrame(summary)

    # Strict target:
    # natural yellow + healthy controls should mostly stay below 15%
    acceptable = summary_df[
        (summary_df["negative_p75_percent"] < 12) &
        (summary_df["negative_p90_percent"] < 20) &
        (summary_df["false_high_rate"] < 0.10)
    ].copy()

    if len(acceptable) > 0:
        best_threshold = float(acceptable.iloc[0]["threshold"])
    else:
        # If nothing passes, choose the strictest threshold.
        best_threshold = float(summary_df["threshold"].max())

    best_bleach_threshold = min(best_threshold + 0.04, 0.95)

    print("Threshold summary:")
    display(summary_df)

    print("Best chlorosis threshold:", best_threshold)
    print("Best bleach threshold:", best_bleach_threshold)

    return best_threshold, best_bleach_threshold


BEST_CHLOROSIS_THRESHOLD, BEST_BLEACH_THRESHOLD = pick_best_threshold(sweep_df)

In [ ]:
model = LichenChlorosisModel(
    chlorosis_threshold=BEST_CHLOROSIS_THRESHOLD,
    bleach_threshold=BEST_BLEACH_THRESHOLD,
    min_region_area=80
)

In [ ]:
Best chlorosis threshold: 0.7400000000000001
Best bleach threshold: 0.7800000000000001

Usable as feature needs validation testing